In [1]:
# import packages
import cobra
import cometspy as c
import os
import pandas as pd
import re
import math

# load models
E_WT_iM = cobra.io.read_sbml_model("../../../models/iML1515.xml")
S0 = cobra.io.read_sbml_model("../../../models/STM_v1_0_S0.xml")

# create E0 knockouts
E0_iM = E_WT_iM.copy()
E0_iM.genes.b3939.knock_out()

met_limiting_medium = {
    'EX_lcts_e': 10,
    'EX_met__L_e': .02,
    'EX_pi_e': 10,
    'EX_fe3_e': 10,
    'EX_mn2_e': 10,
    'EX_fe2_e': 10,
    'EX_zn2_e': 10,
    'EX_mg2_e': 10,
    'EX_ca2_e': 10,
    'EX_ni2_e': 10,
    'EX_cu2_e': 10,
    'EX_cobalt2_e': 10,
    'EX_mobd_e': 10,
    'EX_so4_e': 10,
    'EX_k_e': 10,
    'EX_cl_e': 10,
    'EX_o2_e': 10,
    'EX_nh4_e': 10}

carbon_limiting_medium = {
    'EX_gal_e': 3,
    'EX_met__L_e': 10,
    'EX_pi_e': 10,
    'EX_fe3_e': 10,
    'EX_mn2_e': 10,
    'EX_fe2_e': 10,
    'EX_zn2_e': 10,
    'EX_mg2_e': 10,
    'EX_ca2_e': 10,
    'EX_ni2_e': 10,
    'EX_cu2_e': 10,
    'EX_cobalt2_e': 10,
    'EX_mobd_e': 10,
    'EX_so4_e': 10,
    'EX_k_e': 10,
    'EX_cl_e': 10,
    'EX_o2_e': 10,
    'EX_nh4_e': 10}

# load functions
def get_gene_ids(model, genes):
    all_gene_names = [gene.name for gene in model.genes]
    all_gene_ids = [gene.id for gene in model.genes]
    names_and_ids = dict(zip(all_gene_ids, all_gene_names))
    gene_ids = [key for key, value in names_and_ids.items() if value in genes]
    return(gene_ids)

def scale_reaction(model: "Model", reaction_id: str, alpha: float, direction='forward'):
    if direction == 'forward':
        model.reactions.get_by_id(reaction_id).lower_bound = 0
        dict_product = dict((k, v) for k, v in model.reactions.get_by_id(reaction_id).metabolites.items() if v >= 0)    # obtain 'end product'
    else:  # reverse reaction
        model.reactions.get_by_id(reaction_id).upper_bound = 0
        dict_product = dict((k, v) for k, v in model.reactions.get_by_id(reaction_id).metabolites.items() if v <= 0)
    if isinstance(alpha, pd.Series):
        alpha = float(alpha)
    for product, unit in dict_product.items():  # scale corresponding metabolite units involved in the reaction
        model.reactions.get_by_id(reaction_id).add_metabolites({product: -unit*(1-1/alpha)}) # only change unit of unidirection metabolite

def separate_reaction(model: "Model", reaction_id: str, alpha: float):
    (lb, ub) = model.reactions.get_by_id(reaction_id).bounds
    rxn_ids = [reaction_id] #? 
    if(lb < 0 and ub !=0): # only perform if reaction is bidirectional
        rev_reaction = model.reactions.get_by_id(reaction_id).copy() # copy of target reaction
        rev_reaction.id = f'{reaction_id}_v1' # redefine id for copy of reaction
        model.add_reactions([rev_reaction]) # add to model
    
        scale_reaction(model, rev_reaction.id, alpha, direction='backward')
        
        rxn_ids.append(rev_reaction.id) # list of id for reaction and reaction_v1
    scale_reaction(model, reaction_id, alpha, direction='forward')
    return(rxn_ids)

# function to scale the stochiometry for all reactions related to a gene
def scale_stochiometry(model: "Model", alpha: float, gene: str, KO = False):
    gene_ids = get_gene_ids(model, gene)
    reactions = [reaction.id for reaction in model.genes.get_by_id(gene_ids[0]).reactions]
    rxn_ids = list()
    for rxn in reactions:
        if KO:
            model.reactions.get_by_id(rxn).knock_out()
        elif (rxn not in rxn_ids):
                rxn_ids.extend(separate_reaction(model, rxn, alpha))# copy of reaction, forward_terminal_change = True
    return(rxn_ids)

def get_alpha_value(model: "Model", start_alpha: float, gene: str, tolerance: float, frac_of_biomass: float):
    test_model = model.copy()
    initial_biomass = test_model.slim_optimize() 
    try:
        value = float(initial_biomass)  # Try converting to float
    except ValueError:
        print("Non-float value detected. No initial biomass.")
    alpha = start_alpha
    low = 2
    high = None

    while True:
        scale_stochiometry(test_model, alpha, gene)
        scaled_biomass = test_model.slim_optimize() 
    
        if math.isclose(scaled_biomass, (initial_biomass * frac_of_biomass), rel_tol=tolerance):
            break
    
        if scaled_biomass > (initial_biomass * frac_of_biomass):  
            low = alpha
            alpha = alpha * 2
        else:  
            high = alpha
            alpha = (low + high) / 2
            
        test_model = model.copy()
    
    return(alpha)

def make_scaled_model(model: "Model", alpha: float, gene: str):
    new_model = model.copy()
    scale_stochiometry(new_model, alpha, gene)
    return(new_model)

def pivot_long_fluxes(dataframe):
    df_long = pd.melt(
        dataframe,
        id_vars=['cycle', 'x', 'y'],
        var_name='reaction',
        value_name='flux'
    )
    
    if df_long[['x', 'y']].nunique().eq(1).all() and \
       (df_long['x'].iloc[0] == 1 and df_long['y'].iloc[0] == 1):
        df_long = df_long.drop(columns=['x', 'y'])

    df_long = df_long.drop_duplicates()

    return(df_long)

def fix_biomass_data(dataframe):
    
    new_name = 'biomass'  # or whatever you want to rename it to
    other_col = [col for col in dataframe.columns if col != 'cycle'][0]
    df = dataframe.rename(columns={other_col: new_name})

    return(df)

def drop_x_y(dataframe):
    if (dataframe['x'].eq(1).all()) and (dataframe['y'].eq(1).all()):
        df = dataframe.drop(columns=['x', 'y'])
    df = df.drop_duplicates()

    return(df)

def run_standard_monoculture_simulation(model: "Model", carbon: str, carbon_mmol: float, auxotrophies: str, auxo_mmol: float, max_cycles: int):
    ident = model.id
    
    comets_model = c.model(model)
    comets_model.open_exchanges()
    
    comets_model.initial_pop = [0, 0, 1.e-8]
    comets_model.obj_style = "MAX_OBJECTIVE_MIN_TOTAL"
    
    test_tube = c.layout()
    test_tube.add_model(comets_model)
    
    base_nutrients = ['ca2_e', 'cl_e', 'cobalt2_e', 'cu2_e', 'fe2_e', 'fe3_e', 'h_e', 'k_e', 'h2o_e', 'mg2_e',
             'mn2_e', 'mobd_e', 'na1_e', 'ni2_e', 'nh4_e', 'o2_e', 'pi_e', 'so4_e', 'zn2_e']
    for nutrient in base_nutrients:
        test_tube.set_specific_metabolite(nutrient, 1000)
        test_tube.set_specific_static(nutrient, 1000)
    test_tube.set_specific_metabolite(carbon, carbon_mmol)
    if auxotrophies != "none":
        test_tube.set_specific_metabolite(auxotrophies, auxo_mmol)
    
    sim_params = c.params()
    sim_params.set_param("defaultKm", 0.00001) # M 
    sim_params.set_param("defaultVmax", 10) #mmol/gDw/hr
    sim_params.set_param("maxCycles", max_cycles)
    sim_params.set_param("timeStep", 1)
    sim_params.set_param("writeMediaLog", True)
    sim_params.set_param("MediaLogRate", 1)
    sim_params.set_param("writeFluxLog", True)
    sim_params.set_param("FluxLogRate", 1)
    
    comp_assay = c.comets(test_tube, sim_params)
    comp_assay.run()

    return([fix_biomass_data(comp_assay.total_biomass), pivot_long_fluxes(comp_assay.fluxes_by_species[ident]), drop_x_y(comp_assay.media)])

def run_standard_coculture_simulation(model1: "Model", model2: "Model", carbon: str, carbon_mmol: float, max_cycles: int):
    id1 = model1.id
    id2 = model2.id
    comets1 = c.model(model1)
    comets2 = c.model(model2)
    comets1.open_exchanges()
    comets2.open_exchanges()
    
    comets1.initial_pop = [0, 0, 1.e-8]
    comets2.initial_pop = [0, 0, 1.e-8]
    comets1.obj_style = "MAX_OBJECTIVE_MIN_TOTAL" #this is for pFBA - MAX_OBJECTIVE_FLUX for fba
    comets2.obj_style =  "MAX_OBJECTIVE_MIN_TOTAL"
    
    test_tube = c.layout([comets1, comets2])
    
    base_nutrients = ["ca2_e", "cl_e", "cobalt2_e", "cu2_e","fe2_e", "fe3_e", "k_e","mg2_e",
          "mn2_e", "mobd_e", "ni2_e", "o2_e", "pi_e", "nh4_e", "so4_e", "zn2_e"]
    for nutrient in base_nutrients:
        test_tube.set_specific_metabolite(nutrient, 1000)
    test_tube.set_specific_metabolite(carbon, carbon_mmol)
    
    p = c.params()
    p.set_param("defaultKm", 0.00001) # M 
    p.set_param("defaultVmax", 10) #mmol/gDw/hr
    p.set_param("maxCycles", max_cycles)
    p.set_param("timeStep", 1)
    p.set_param("writeMediaLog", True)
    p.set_param("MediaLogRate", 1)
    p.set_param("writeFluxLog", True)
    p.set_param("FluxLogRate", 1)
    
    comp_assay = c.comets(test_tube, p)
    comp_assay.run()

    return([comp_assay.total_biomass, comp_assay.fluxes_by_species[id1], comp_assay.fluxes_by_species[id2], comp_assay.media])

num_single_repression_cycles = 200
S0_medium = carbon_limiting_medium
E0_medium = met_limiting_medium

# run wild type simulations
S_WT = run_standard_monoculture_simulation(model = S0, carbon = "gal_e", carbon_mmol = 0.000278, auxotrophies = "none", auxo_mmol = 0, max_cycles = num_single_repression_cycles)
for df in S_WT:
    df['gene'] = "WT"
for df in S_WT:
    df['strain'] = "S"
for df in S_WT:
    df['interaction'] = "monoculture"

E_WT = run_standard_monoculture_simulation(model = E0_iM, carbon = "lcts_e", carbon_mmol = 0.000278, auxotrophies = "met__L_e", auxo_mmol = 0.000278 / 50, max_cycles = num_single_repression_cycles)
for df in E_WT:
    df['gene'] = "WT"
for df in E_WT:
    df['strain'] = "E"
for df in E_WT:
    df['interaction'] = "monoculture"

WT = run_standard_coculture_simulation(model1 = E0_iM, model2 = S0, carbon = "lcts_e", carbon_mmol = 0.000278, max_cycles = num_single_repression_cycles)
WT[0] = WT[0].melt(
    id_vars=['cycle'],
    value_vars=[col for col in WT[0].columns if col.startswith('iM') or col.startswith('STM')],
    var_name='strain',
    value_name='biomass'
)

WT[0]["gene"] = "WT"
WT[0]["interaction"] = "coculture"

WT[1] = pivot_long_fluxes(WT[1])
WT[1]["strain"] = "E"
WT[1]["gene"] = "WT"
WT[1]["interaction"] = "coculture"

WT[2] = pivot_long_fluxes(WT[2])
WT[2]["strain"] = "S"
WT[2]["gene"] = "WT"
WT[2]["interaction"] = "coculture"

WT[3] = drop_x_y(WT[3])
WT[3]["strain"] = "ES"
WT[3]["gene"] = "WT"
WT[3]["interaction"] = "coculture"

biomass = pd.concat([S_WT[0], E_WT[0], WT[0]])
fluxes = pd.concat([S_WT[1], E_WT[1], WT[1], WT[2]])
metabolites = pd.concat([S_WT[2], E_WT[2], WT[3]])

biomass.to_csv('../../../model-data/alpha-values/WT-biomasses-1-step.csv')
fluxes.to_csv('../../../model-data/alpha-values/WT-fluxes-1-step.csv')
metabolites.to_csv('../../../model-data/alpha-values/WT-metabolites-1-step.csv')

building empty layout model
models will need to be added with layout.add_model()

Running COMETS simulation ...

Debug Here ...
Done!
building empty layout model
models will need to be added with layout.add_model()

Running COMETS simulation ...

Debug Here ...
Done!

Running COMETS simulation ...

Debug Here ...
Done!


In [3]:
fluxes

,cycle,reaction,flux,gene,strain,interaction
0,1,3HAD180,0.002368,WT,S,monoculture
1,2,3HAD180,0.002368,WT,S,monoculture
2,3,3HAD180,0.002368,WT,S,monoculture
3,4,3HAD180,0.002368,WT,S,monoculture
4,5,3HAD180,0.002368,WT,S,monoculture
...,...,...,...,...,...,...
508995,196,OA4VL_ST,0.000000,WT,S,coculture
508996,197,OA4VL_ST,0.000000,WT,S,coculture
508997,198,OA4VL_ST,0.000000,WT,S,coculture
508998,199,OA4VL_ST,0.000000,WT,S,coculture
